In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# **Oil Sands Analysis Project**

# Goal: 
The idea of the project is to get an idea of Alberta oil production vs WTI (West-Texas intermediate) prices. On paper, it seems very simple: if global oil prices are higher, Alberta should produce more. The issue with this idea is that many factors are in play when trying to increase oil production in Alberta. These can be weather, taxes, emissions, and so on and so forth.    

### Data Sources
Our sources come from the St. Louis Federal Reserve and the Government of Alberta website

In [3]:
#The St. Louis Fed data is separated by date and price
#The Alberta data set is separated by non vs. conventional oil. Non-conventional is MOSTLY oilsands products (bitumen and synthetic)
#We will first do all of the cleaning on the St. Louis Fed data and then clean the Al
data = pd.read_csv("data/DCOILWTICO.csv") 
data_alb = pd.read_csv("data/alberta_production_by_type.csv")

In [4]:
data.head() 

,observation_date,DCOILWTICO
0,1986-01-02,25.56
1,1986-01-03,26.00
2,1986-01-06,26.53
3,1986-01-07,25.85
4,1986-01-08,25.87


In [106]:
data_alb.head()

,Date,Value,Series,labels
0,2007-01-01,2707743.4,Conventional Oil,2007-01-01T00:00:00
1,2007-02-01,2450594.2,Conventional Oil,2007-02-01T00:00:00
2,2007-03-01,2703836.8,Conventional Oil,2007-03-01T00:00:00
3,2007-04-01,2567241.4,Conventional Oil,2007-04-01T00:00:00
4,2007-05-01,2614545.2,Conventional Oil,2007-05-01T00:00:00


In [107]:
data.shape

(10513, 2)

In [108]:
data_alb.shape

(460, 4)

In [109]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10513 entries, 0 to 10512
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  10513 non-null  object 
 1   DCOILWTICO        10143 non-null  float64
dtypes: float64(1), object(1)
memory usage: 164.4+ KB


In [110]:
data_alb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460 entries, 0 to 459
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    460 non-null    object 
 1   Value   460 non-null    float64
 2   Series  460 non-null    object 
 3   labels  460 non-null    object 
dtypes: float64(1), object(3)
memory usage: 14.5+ KB


In [111]:
data.columns = ["date", "price"]
data["date"] = pd.to_datetime(data["date"])
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10513 entries, 0 to 10512
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    10513 non-null  datetime64[ns]
 1   price   10143 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 164.4 KB


In [112]:
data_alb.columns = ["date", "produced", "type", "labels"]
data_alb["date"] = pd.to_datetime(data_alb["date"])

In [113]:
data_alb.head()

,date,produced,type,labels
0,2007-01-01,2707743.4,Conventional Oil,2007-01-01T00:00:00
1,2007-02-01,2450594.2,Conventional Oil,2007-02-01T00:00:00
2,2007-03-01,2703836.8,Conventional Oil,2007-03-01T00:00:00
3,2007-04-01,2567241.4,Conventional Oil,2007-04-01T00:00:00
4,2007-05-01,2614545.2,Conventional Oil,2007-05-01T00:00:00


In [114]:
data_alb = data_alb.drop(columns=["labels"])

In [115]:
data_alb.head()

,date,produced,type
0,2007-01-01,2707743.4,Conventional Oil
1,2007-02-01,2450594.2,Conventional Oil
2,2007-03-01,2703836.8,Conventional Oil
3,2007-04-01,2567241.4,Conventional Oil
4,2007-05-01,2614545.2,Conventional Oil


Data exploration

In [116]:
data.isnull().sum()

date       0
price    370
dtype: int64

In [117]:
data_alb.isnull().sum()

date        0
produced    0
type        0
dtype: int64

In [118]:
#If a data point is missing, take on the value of the previous day's
data["price"] = data["price"].ffill().bfill()

In [119]:
data.isnull().sum()

date     0
price    0
dtype: int64

In [120]:
data.describe()

,date,price
count,10513,10513.000000
mean,2006-02-24 19:11:56.712641536,48.326903
min,1986-01-02 00:00:00,-36.980000
25%,1996-01-30 00:00:00,20.340000
50%,2006-02-24 00:00:00,43.210000
75%,2016-03-23 00:00:00,71.280000
max,2026-04-20 00:00:00,145.310000
std,NaN,29.448196


In [121]:
data_alb.describe()

,date,produced
count,460,4.600000e+02
mean,2016-07-16 12:50:05.217391360,7.114597e+06
min,2007-01-01 00:00:00,1.982695e+06
25%,2011-10-01 00:00:00,2.664248e+06
50%,2016-07-16 12:00:00,4.181578e+06
75%,2021-05-01 00:00:00,1.226469e+07
max,2026-02-01 00:00:00,1.846083e+07
std,NaN,5.201103e+06


In [122]:
data_alb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460 entries, 0 to 459
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      460 non-null    datetime64[ns]
 1   produced  460 non-null    float64       
 2   type      460 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 10.9+ KB


In [123]:
#We should have exaclty 230 for each type of oil
data_alb["type"].value_counts()

type
Conventional Oil        230
Non-Conventional Oil    230
Name: count, dtype: int64

In [124]:
#Now we do a pivot so the conventional/non-conventional are side by side and not merged into one column
alb_wide = data_alb.pivot(index="date", columns="type", values="produced")
alb_wide = alb_wide.reset_index()
alb_wide.head()

type,date,Conventional Oil,Non-Conventional Oil
0,2007-01-01,2707743.4,5604533.2
1,2007-02-01,2450594.2,5414892.8
2,2007-03-01,2703836.8,6082110.5
3,2007-04-01,2567241.4,5463210.8
4,2007-05-01,2614545.2,5460600.2


In [125]:
data.to_csv("data/wti_cleaned.csv", index=False)

In [126]:
#The cleaned table still had some missing fields, manually check said fields
data.loc[33]

date     1986-02-18 00:00:00
price                   14.7
Name: 33, dtype: object

In [127]:
alb_wide.to_csv("data/alberta_production_wide.csv", index=False)

In [128]:
data_alb.to_csv("data/alberta_production_long.csv", index=False)